# 04a — Exposition : Bâtiments, Population & Valeurs foncières

Ce notebook prépare la couche d'**exposition** pour l'analyse du risque inondation.

| Source | Contenu | Résolution |
|--------|---------|------------|
| **BD TOPO IGN** | Emprise, hauteur, usage des bâtiments | Bâtiment |
| **OSM** | Bâtiments avec attributs d'usage | Bâtiment |
| **FILOSOFI INSEE** | Population + revenus à la maille 200m | 200m |
| **DVF DGFiP** | Valeurs transactions immobilières | Parcelle |

**Département pilote : Hérault (34)**

In [ ]:
# !pip install requests geopandas pandas matplotlib folium osmnx pyarrow

import os, json, warnings
import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display
warnings.filterwarnings('ignore')

# ── Constantes ────────────────────────────────────────────────────────────────
DEPT_CODE  = "34"
DEPT_NAME  = "Hérault"
BBOX       = [2.85, 43.21, 3.98, 43.95]   # lon_min, lat_min, lon_max, lat_max
BBOX_MPL   = (3.80, 43.55, 3.92, 43.65)   # extrait Montpellier centre
DATA_DIR   = "../../data/exposure"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_DATAGOUV = "https://www.data.gouv.fr/api/1"

# Correspondance BD TOPO NATURE → catégorie JRC
BDTOPO_TO_JRC = {
    "Indifférenciée"                    : "residential",
    "Résidentiel"                       : "residential",
    "Industriel, agricole ou commercial": "industrial",
    "Commercial et services"            : "commercial",
    "Industriel"                        : "industrial",
    "Agricole"                          : "agriculture",
    "Sportif"                           : "commercial",
    "Religieux"                         : "commercial",
    "Administratif ou militaire"        : "commercial",
    "Serre agricole"                    : "agriculture",
}

# Valeurs max dommage JRC (€/m² de plancher) — France
JRC_MAX_DAMAGE = {
    "residential" : 1010,
    "commercial"  : 730,
    "industrial"  : 460,
    "agriculture" : 1.5,
    "transport"   : 82,
}

print(f"✅ Setup OK — Département : {DEPT_NAME} ({DEPT_CODE})")
print(f"📁 Dossier données : {DATA_DIR}")

---
## 1. BD TOPO IGN — Bâtiments

In [ ]:
# ── 1.1 Recherche de la ressource GeoParquet Hérault sur data.gouv.fr ────────
BDTOPO_DATASET_ID = "61489083dc4223219e50cc35"

r = requests.get(f"{BASE_DATAGOUV}/datasets/{BDTOPO_DATASET_ID}/", timeout=15)
print(f"Status BD TOPO API : {r.status_code}")

bdtopo_url = None
if r.status_code == 200:
    resources = r.json().get('resources', [])
    print(f"Nombre total de ressources : {len(resources)}")
    # Recherche ressource Hérault (034) BATIMENT GeoParquet
    for res in resources:
        title = res.get('title', '').lower()
        fmt   = res.get('format', '').lower()
        if ('034' in title or 'herault' in title or 'hérault' in title):
            if 'batiment' in title or 'bâtiment' in title or fmt in ['parquet','geoparquet']:
                print(f"  ✅ [{fmt:12s}] {res['title'][:70]}")
                print(f"       {res.get('url','')[:90]}")
                bdtopo_url = res.get('url')
                break
    if not bdtopo_url:
        print("  ⚠️  Pas de ressource GeoParquet directe trouvée.")
        print("  → Alternative WFS IGN : https://data.geopf.fr/wfs/ows?SERVICE=WFS&REQUEST=GetFeature")
        print(f"     &TYPENAME=BDTOPO_V3:batiment&BBOX={','.join(map(str,BBOX))}")

In [ ]:
# ── 1.2 Chargement via WFS IGN (Géoplateforme) ────────────────────────────────
# API WFS de la Géoplateforme IGN (remplace Géoportail)

WFS_IGN = "https://data.geopf.fr/wfs/ows"

# Requête sur une zone restreinte (Montpellier centre) pour test
bbox_str = f"{BBOX_MPL[0]},{BBOX_MPL[1]},{BBOX_MPL[2]},{BBOX_MPL[3]},EPSG:4326"
params_wfs = {
    "SERVICE"     : "WFS",
    "VERSION"     : "2.0.0",
    "REQUEST"     : "GetFeature",
    "TYPENAME"    : "BDTOPO_V3:batiment",
    "BBOX"        : bbox_str,
    "OUTPUTFORMAT": "application/json",
    "COUNT"       : 500
}

gdf_bdtopo = None
try:
    r_wfs = requests.get(WFS_IGN, params=params_wfs, timeout=30)
    print(f"Status WFS IGN Bâtiments : {r_wfs.status_code}")
    if r_wfs.status_code == 200 and r_wfs.headers.get('content-type','').startswith('application'):
        gdf_bdtopo = gpd.GeoDataFrame.from_features(r_wfs.json().get('features', []), crs='EPSG:4326')
        print(f"Bâtiments chargés : {len(gdf_bdtopo)} (Montpellier centre)")
        if not gdf_bdtopo.empty:
            print(f"Colonnes : {list(gdf_bdtopo.columns)}")
            display(gdf_bdtopo[['nature','usage1','hauteur','nb_logts','nb_etages']].head(5))
except Exception as e:
    print(f"⚠️  WFS non disponible : {e}")

In [ ]:
# ── 1.3 Données de démonstration si WFS indisponible ─────────────────────────
if gdf_bdtopo is None or gdf_bdtopo.empty:
    print("⚠️  Génération de données de démonstration (WFS non disponible)")
    from shapely.geometry import box
    import random
    random.seed(42)
    np.random.seed(42)
    natures = ["Indifférenciée","Résidentiel","Commercial et services",
                "Industriel, agricole ou commercial","Agricole"]
    weights = [0.50, 0.25, 0.12, 0.08, 0.05]
    n = 300
    lons = np.random.uniform(BBOX_MPL[0], BBOX_MPL[2], n)
    lats = np.random.uniform(BBOX_MPL[1], BBOX_MPL[3], n)
    ws   = np.random.uniform(0.0001, 0.0004, n)
    hs   = np.random.uniform(0.0001, 0.0003, n)
    geoms = [box(lons[i], lats[i], lons[i]+ws[i], lats[i]+hs[i]) for i in range(n)]
    gdf_bdtopo = gpd.GeoDataFrame({
        "nature"    : np.random.choice(natures, n, p=weights),
        "usage1"    : np.random.choice(["Résidentiel","Tertiaire","Industriel","Agricole"], n),
        "hauteur"   : np.random.uniform(3, 25, n).round(1),
        "nb_logts"  : np.random.randint(0, 50, n),
        "nb_etages" : np.random.randint(1, 8, n),
        "geometry"  : geoms
    }, crs="EPSG:4326")
    print(f"✅ Données démo créées : {len(gdf_bdtopo)} bâtiments")

In [ ]:
# ── 1.4 Statistiques BD TOPO ──────────────────────────────────────────────────
# Projection Lambert 93 pour calcul de surfaces
gdf_l93 = gdf_bdtopo.to_crs('EPSG:2154')
gdf_l93['area_m2'] = gdf_l93.geometry.area

# Mapping vers catégorie JRC
gdf_l93['jrc_category'] = gdf_l93['nature'].map(BDTOPO_TO_JRC).fillna('residential')

# Agrégation par nature
stats = gdf_l93.groupby('jrc_category').agg(
    nb_batiments  = ('area_m2', 'count'),
    surface_tot_m2= ('area_m2', 'sum'),
    surface_moy_m2= ('area_m2', 'mean'),
    hauteur_moy_m = ('hauteur', 'mean')
).round(1)
stats['surface_tot_ha'] = (stats['surface_tot_m2'] / 10000).round(2)
print("Statistiques BD TOPO par catégorie JRC :")
display(stats)

In [ ]:
# ── 1.5 Visualisation BD TOPO ─────────────────────────────────────────────────
COLOR_MAP = {
    "residential": "#e74c3c",
    "commercial"  : "#e67e22",
    "industrial"  : "#9b59b6",
    "agriculture" : "#27ae60",
    "transport"   : "#3498db",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Carte des bâtiments colorés par catégorie JRC
ax1 = axes[0]
for cat, color in COLOR_MAP.items():
    subset = gdf_bdtopo[gdf_bdtopo['nature'].map(BDTOPO_TO_JRC).fillna('residential') == cat]
    if not subset.empty:
        subset.plot(ax=ax1, color=color, alpha=0.7, label=cat)
ax1.set_title(f"BD TOPO — Bâtiments par catégorie\n(Montpellier centre)", fontsize=11)
ax1.legend(fontsize=8, loc='upper right')
ax1.set_xlabel("Longitude"); ax1.set_ylabel("Latitude")
ax1.grid(alpha=0.2)

# Graphique en barres : nb bâtiments + surface par catégorie
ax2 = axes[1]
cats = list(stats.index)
colors = [COLOR_MAP.get(c, 'gray') for c in cats]
bars = ax2.bar(cats, stats['nb_batiments'], color=colors, alpha=0.8)
for bar, val in zip(bars, stats['nb_batiments']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(int(val)), ha='center', va='bottom', fontsize=9)
ax2.set_title("Nombre de bâtiments par catégorie", fontsize=11)
ax2.set_ylabel("Nombre de bâtiments")
ax2.tick_params(axis='x', rotation=20)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/bdtopo_batiments_herault.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Carte sauvegardée")

---
## 2. OpenStreetMap — Bâtiments via Overpass API

In [ ]:
# ── 2.1 Téléchargement OSM via Overpass API ───────────────────────────────────
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

# Requête Overpass : bâtiments sur Montpellier centre
query = f"""
[out:json][timeout:60][bbox:{BBOX_MPL[1]},{BBOX_MPL[0]},{BBOX_MPL[3]},{BBOX_MPL[2]}];
(
  way["building"];
  relation["building"]["type"="multipolygon"];
);
out body;
>;
out skel qt;
"""

gdf_osm = None
try:
    r_osm = requests.post(OVERPASS_URL, data=query, timeout=60)
    print(f"Status Overpass : {r_osm.status_code}")
    if r_osm.status_code == 200:
        data = r_osm.json()
        print(f"Éléments OSM retournés : {len(data.get('elements', []))}")
except Exception as e:
    print(f"⚠️  Overpass non disponible : {e}")

# Alternative recommandée avec osmnx
osmnx_code = """
import osmnx as ox

# Téléchargement bâtiments OSM
gdf_osm = ox.features_from_bbox(
    bbox=(BBOX_MPL[3], BBOX_MPL[1], BBOX_MPL[2], BBOX_MPL[0]),
    tags={'building': True}
)
# Garder seulement les polygones
gdf_osm = gdf_osm[gdf_osm.geometry.geom_type.isin(['Polygon','MultiPolygon'])]
print(f"Bâtiments OSM : {len(gdf_osm)}")
"""
print("\nCode osmnx (décommenter après pip install osmnx) :")
print(osmnx_code)

In [ ]:
# ── 2.2 Classification OSM → catégories JRC ───────────────────────────────────
def classify_osm_building(building_tag: str) -> str:
    """Mappe le tag OSM 'building' vers une catégorie JRC."""
    tag = str(building_tag).lower().strip()
    residential = {'yes','house','residential','apartments','detached','semidetached_house',
                   'terrace','bungalow','dormitory','houseboat'}
    commercial  = {'commercial','retail','supermarket','shop','office','hotel','bank',
                   'hospital','school','university','church','cathedral','museum','theatre'}
    industrial  = {'industrial','warehouse','factory','manufacture','storage_tank','silo'}
    agriculture = {'farm','barn','greenhouse','stable','sty','cowshed','farm_auxiliary'}
    transport   = {'garage','carport','parking','train_station','bus_station','hangar'}

    if tag in residential : return 'residential'
    if tag in commercial  : return 'commercial'
    if tag in industrial  : return 'industrial'
    if tag in agriculture : return 'agriculture'
    if tag in transport   : return 'transport'
    return 'residential'  # défaut

# Démonstration sur des exemples
test_tags = ['house','apartments','commercial','warehouse','barn','yes','office','factory']
print(f"{'Tag OSM':<20} → {'Catégorie JRC':<15}")
print("-" * 38)
for tag in test_tags:
    print(f"  {tag:<18} → {classify_osm_building(tag):<15}")

In [ ]:
# ── 2.3 Comparaison BD TOPO vs OSM ────────────────────────────────────────────
comparison = pd.DataFrame([
    {"Critère": "Couverture France",       "BD TOPO IGN"   : "✅ Nationale complète",      "OSM"            : "✅ Très bonne en zones urbaines"},
    {"Critère": "Résolution géométrie",    "BD TOPO IGN"   : "✅ Haute précision cadastre", "OSM"            : "⚠️  Variable"},
    {"Critère": "Attribut usage",          "BD TOPO IGN"   : "✅ NATURE + USAGE1",          "OSM"            : "⚠️  Partiel (building=*)"},
    {"Critère": "Hauteur bâtiment",        "BD TOPO IGN"   : "✅ Champ HAUTEUR",            "OSM"            : "⚠️  building:levels (parfois)"},
    {"Critère": "Nb logements",            "BD TOPO IGN"   : "✅ NB_LOGTS",                 "OSM"            : "❌ Non disponible"},
    {"Critère": "Mise à jour",             "BD TOPO IGN"   : "Annuelle",                    "OSM"            : "Continue (communautaire)"},
    {"Critère": "Licence",                 "BD TOPO IGN"   : "Licence Ouverte Etalab",      "OSM"            : "ODbL"},
    {"Critère": "Format",                  "BD TOPO IGN"   : "GeoParquet, SHP, WFS",        "OSM"            : "GeoJSON, PBF, Parquet"},
    {"Critère": "Recommandation",          "BD TOPO IGN"   : "🥇 Préférable pour France",   "OSM"            : "🥈 Utile pour zones non couvertes"},
])
print("Comparaison BD TOPO IGN vs OpenStreetMap :")
display(comparison)

---
## 3. Préparation de la couche d'exposition

In [ ]:
# ── 3.1 Fonction de préparation de la couche d'exposition ─────────────────────
def prepare_exposure_layer(gdf: gpd.GeoDataFrame,
                            nature_col: str = 'nature',
                            hauteur_col: str = 'hauteur',
                            nb_etages_col: str = 'nb_etages') -> gpd.GeoDataFrame:
    """
    Prépare la couche d'exposition pour le calcul des dommages inondation.

    Sortie : GeoDataFrame enrichi avec :
      - jrc_category       : catégorie JRC (residential, commercial, ...)
      - area_m2            : surface emprise au sol (m²)
      - nb_etages          : nombre d'étages estimé
      - floor_area_m2      : surface de plancher totale (m²)
      - max_dmg_eur_m2     : valeur max dommage JRC (€/m²)
      - max_damage_total_eur: valeur max dommage totale (€)
    """
    gdf = gdf.copy()

    # 1. Projection Lambert 93 pour surfaces
    if gdf.crs and gdf.crs.to_epsg() != 2154:
        gdf = gdf.to_crs('EPSG:2154')

    # 2. Surface emprise
    gdf['area_m2'] = gdf.geometry.area.round(1)

    # 3. Catégorie JRC
    gdf['jrc_category'] = gdf[nature_col].map(BDTOPO_TO_JRC).fillna('residential')

    # 4. Nombre d'étages (défaut 2 si inconnu)
    if nb_etages_col in gdf.columns:
        gdf['nb_etages'] = pd.to_numeric(gdf[nb_etages_col], errors='coerce').fillna(2).clip(1, 20).astype(int)
    else:
        # Estimation depuis hauteur si disponible (3m/étage)
        if hauteur_col in gdf.columns:
            h = pd.to_numeric(gdf[hauteur_col], errors='coerce').fillna(6)
            gdf['nb_etages'] = (h / 3).clip(1, 20).round().astype(int)
        else:
            gdf['nb_etages'] = 2

    # 5. Surface de plancher
    gdf['floor_area_m2'] = (gdf['area_m2'] * gdf['nb_etages']).round(1)

    # 6. Valeur max dommage JRC
    gdf['max_dmg_eur_m2']      = gdf['jrc_category'].map(JRC_MAX_DAMAGE)
    gdf['max_damage_total_eur'] = (gdf['floor_area_m2'] * gdf['max_dmg_eur_m2']).round(0)

    return gdf[['geometry','jrc_category','area_m2','nb_etages',
                'floor_area_m2','max_dmg_eur_m2','max_damage_total_eur']]


# Application
gdf_exposure = prepare_exposure_layer(gdf_bdtopo)
print(f"Couche exposition : {len(gdf_exposure)} bâtiments")
display(gdf_exposure.head(5))
print(f"\nValeur totale exposée (zone test) : {gdf_exposure['max_damage_total_eur'].sum()/1e6:.1f} M€")

In [ ]:
# ── 3.2 Statistiques de la couche exposition ──────────────────────────────────
expo_stats = gdf_exposure.groupby('jrc_category').agg(
    nb_batiments         = ('area_m2','count'),
    surface_tot_m2       = ('area_m2','sum'),
    floor_area_tot_m2    = ('floor_area_m2','sum'),
    valeur_max_total_eur = ('max_damage_total_eur','sum')
).round(0)
expo_stats['valeur_max_M_eur'] = (expo_stats['valeur_max_total_eur'] / 1e6).round(2)
expo_stats['part_%'] = (expo_stats['valeur_max_total_eur'] / expo_stats['valeur_max_total_eur'].sum() * 100).round(1)

print("Résumé de la couche d'exposition :")
display(expo_stats[['nb_batiments','floor_area_tot_m2','valeur_max_M_eur','part_%']])

# Graphique en secteurs
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = [COLOR_MAP.get(c,'gray') for c in expo_stats.index]

axes[0].pie(expo_stats['nb_batiments'], labels=expo_stats.index, colors=colors,
            autopct='%1.0f%%', startangle=90)
axes[0].set_title("Répartition bâtiments\n(par catégorie)")

axes[1].pie(expo_stats['valeur_max_total_eur'], labels=expo_stats.index, colors=colors,
            autopct='%1.0f%%', startangle=90)
axes[1].set_title("Répartition valeur exposée\n(par catégorie)")

plt.suptitle(f"Exposition — {DEPT_NAME} (zone test Montpellier)", y=1.02)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/exposition_repartition.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 4. FILOSOFI — Population & Revenus (INSEE)

In [ ]:
# ── 4.1 Accès FILOSOFI 200m ───────────────────────────────────────────────────
# FILOSOFI 2019 — Carroyage 200m
# Téléchargement : https://www.insee.fr/fr/statistiques/7655475

FILOSOFI_URL = "https://www.data.gouv.fr/fr/datasets/r/d8f88428-caef-43f6-adc4-87e17d0cee4a"

filosofi_info = {
    "Nom"          : "FILOSOFI 2019 — Carroyage 200m",
    "Source"       : "INSEE",
    "Résolution"   : "Grille 200m × 200m (RGF93 Lambert 93)",
    "Variables clés": [
        "Ind       — Population totale",
        "Men       — Nombre de ménages",
        "Men_pauv  — Ménages pauvres (<60% médiane)",
        "MedRevnu  — Revenu médian disponible par UC (€/an)",
        "Men_1ind  — Ménages unipersonnels (vulnérabilité sociale)",
        "Ind_snsa  — Personnes sans abri (si disponible)",
    ],
    "Format"       : "GeoPackage (.gpkg) ou Parquet",
    "Usage risque" : "Vulnérabilité sociale, population exposée, pertes indirectes"
}

# Code de chargement
filosofi_code = f"""
import geopandas as gpd

# Télécharger et charger FILOSOFI 200m
# Option A — GeoPackage national
gdf_filo = gpd.read_file("Filosofi2019_carreaux_200m_gpkg.zip")

# Filtrer sur le département Hérault (bbox Lambert 93)
gdf_filo = gdf_filo.cx[600000:850000, 6200000:6400000]  # approx Hérault en L93

# Variables d'intérêt pour l'inondation
cols = ['Ind','Men','Men_pauv','MedRevnu','Men_1ind','geometry']
gdf_filo = gdf_filo[cols].copy()

# Indicateur de vulnérabilité sociale (0–1)
gdf_filo['vuln_sociale'] = (
    gdf_filo['Men_pauv'] / gdf_filo['Men'].clip(lower=1)
).clip(0,1)

print(f"Mailles FILOSOFI Hérault : {{len(gdf_filo)}}")
print(f"Population totale estimée : {{gdf_filo['Ind'].sum():,.0f}}")
"""

for k, v in filosofi_info.items():
    if isinstance(v, list):
        print(f"\n{k}:")
        for item in v: print(f"  - {item}")
    else:
        print(f"{k}: {v}")
print("\nCode de chargement (à décommenter) :")
print(filosofi_code)

In [ ]:
# ── 4.2 Démonstration FILOSOFI (données synthétiques) ────────────────────────
# Grille 200m synthétique sur le BBOX Hérault pour démonstration
from shapely.geometry import box

lon_range = np.arange(BBOX[0], BBOX[2], 0.003)
lat_range = np.arange(BBOX[1], BBOX[3], 0.002)
np.random.seed(123)
n_cells = len(lon_range) * len(lat_range)

cells, lons_c, lats_c = [], [], []
for lo in lon_range:
    for la in lat_range:
        cells.append(box(lo, la, lo+0.003, la+0.002))
        lons_c.append(lo + 0.0015)
        lats_c.append(la + 0.001)

# Simulation d'une distribution spatiale réaliste (Montpellier dense)
lons_arr = np.array(lons_c)
lats_arr = np.array(lats_c)
dist_mpl = np.sqrt((lons_arr - 3.876)**2 + (lats_arr - 43.611)**2)
pop_base  = np.exp(-dist_mpl * 8) * 500 + np.random.exponential(20, len(cells))

gdf_filo = gpd.GeoDataFrame({
    'Ind'       : pop_base.round(0).astype(int),
    'Men'       : (pop_base / 2.2).round(0).astype(int),
    'Men_pauv'  : (pop_base * 0.12 / 2.2).round(0).astype(int),
    'MedRevnu'  : (22000 - dist_mpl * 10000 + np.random.normal(0, 2000, len(cells))).clip(8000, 45000).round(0),
    'geometry'  : cells
}, crs='EPSG:4326')

gdf_filo['vuln_sociale'] = (gdf_filo['Men_pauv'] / gdf_filo['Men'].clip(lower=1)).clip(0,1)

print(f"Grille FILOSOFI démo : {len(gdf_filo)} mailles")
print(f"Population totale estimée (démo) : {gdf_filo['Ind'].sum():,.0f}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gdf_filo.plot(column='Ind', ax=axes[0], cmap='YlOrRd', legend=True,
               legend_kwds={'label':'Population (200m)','shrink':0.7})
axes[0].set_title(f"FILOSOFI — Population\n{DEPT_NAME}", fontsize=11)
axes[0].grid(alpha=0.2)

gdf_filo.plot(column='MedRevnu', ax=axes[1], cmap='RdYlGn', legend=True,
               legend_kwds={'label':'Revenu médian (€/an)','shrink':0.7})
axes[1].set_title(f"FILOSOFI — Revenu médian\n{DEPT_NAME}", fontsize=11)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/filosofi_herault.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 5. DVF — Demandes de Valeurs Foncières

In [ ]:
# ── 5.1 Chargement DVF Hérault ────────────────────────────────────────────────
DVF_URL = f"https://files.data.gouv.fr/geo-dvf/latest/csv/2023/departements/{DEPT_CODE}.csv.gz"
print(f"URL DVF : {DVF_URL}")

gdf_dvf = None
try:
    print("Téléchargement DVF...")
    df_dvf = pd.read_csv(DVF_URL, compression='gzip', low_memory=False)
    print(f"DVF chargé : {len(df_dvf)} transactions")
    print(f"Colonnes : {list(df_dvf.columns[:15])}")

    # Filtres : ventes immobilières avec coordonnées
    df_dvf = df_dvf[
        (df_dvf['nature_mutation'] == 'Vente') &
        (df_dvf['longitude'].notna()) &
        (df_dvf['latitude'].notna()) &
        (df_dvf['valeur_fonciere'] > 0) &
        (df_dvf['surface_reelle_bati'] > 5)
    ].copy()

    df_dvf['prix_m2'] = (df_dvf['valeur_fonciere'] / df_dvf['surface_reelle_bati']).round(0)
    df_dvf = df_dvf[(df_dvf['prix_m2'] > 100) & (df_dvf['prix_m2'] < 25000)]

    gdf_dvf = gpd.GeoDataFrame(
        df_dvf,
        geometry=gpd.points_from_xy(df_dvf['longitude'], df_dvf['latitude']),
        crs='EPSG:4326'
    )
    print(f"Transactions nettoyées : {len(gdf_dvf)}")
    print(f"Prix médian/m² : {gdf_dvf['prix_m2'].median():.0f} €/m²")

except Exception as e:
    print(f"⚠️  DVF non disponible ({e}) — génération données démo")
    n_dvf = 1500
    np.random.seed(42)
    lons_d = np.random.uniform(BBOX[0], BBOX[2], n_dvf)
    lats_d = np.random.uniform(BBOX[1], BBOX[3], n_dvf)
    dist_d = np.sqrt((lons_d - 3.876)**2 + (lats_d - 43.611)**2)
    prix   = (4500 - dist_d * 3000 + np.random.normal(0, 500, n_dvf)).clip(1200, 9000)
    surfs  = np.random.exponential(75, n_dvf).clip(15, 400).round(0)
    gdf_dvf = gpd.GeoDataFrame({
        'valeur_fonciere'   : (prix * surfs).round(0),
        'surface_reelle_bati': surfs,
        'prix_m2'           : prix.round(0),
        'type_local'        : np.random.choice(['Appartement','Maison'], n_dvf, p=[0.55,0.45]),
        'code_commune'      : np.random.choice(['34172','34032','34295'], n_dvf),
        'geometry'          : gpd.points_from_xy(lons_d, lats_d)
    }, crs='EPSG:4326')
    print(f"DVF démo : {len(gdf_dvf)} transactions")

In [ ]:
# ── 5.2 Visualisation DVF ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Carte prix/m²
sc = axes[0].scatter(
    gdf_dvf.geometry.x, gdf_dvf.geometry.y,
    c=gdf_dvf['prix_m2'], cmap='RdYlGn_r', s=8, alpha=0.5,
    vmin=gdf_dvf['prix_m2'].quantile(0.05),
    vmax=gdf_dvf['prix_m2'].quantile(0.95)
)
plt.colorbar(sc, ax=axes[0], label='Prix €/m²', shrink=0.8)
axes[0].set_title(f"DVF 2023 — Prix au m²\n{DEPT_NAME}", fontsize=11)
axes[0].grid(alpha=0.2)

# Distribution des prix
axes[1].hist(gdf_dvf['prix_m2'], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(gdf_dvf['prix_m2'].median(), color='red', ls='--', label=f"Médiane : {gdf_dvf['prix_m2'].median():.0f} €/m²")
axes[1].axvline(gdf_dvf['prix_m2'].mean(),   color='orange', ls='--', label=f"Moyenne : {gdf_dvf['prix_m2'].mean():.0f} €/m²")
axes[1].set_title("Distribution des prix au m²", fontsize=11)
axes[1].set_xlabel("€/m²")
axes[1].set_ylabel("Nombre de transactions")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/dvf_herault.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Synthèse — Couche d'exposition combinée

In [ ]:
# ── 6.1 Jonction spatiale BD TOPO + DVF (valeur marchande) ───────────────────
# Idée : enrichir chaque bâtiment avec le prix/m² DVF de sa zone
# → meilleure estimation de la valeur exposée que la valeur JRC forfaitaire

# Reproject tout en WGS84 pour la jointure
gdf_expo_wgs = gdf_exposure.to_crs('EPSG:4326').copy()

# Jointure spatiale : point DVF le plus proche de chaque bâtiment
# (simplification : on utilise les centroïdes)
gdf_expo_wgs['centroid'] = gdf_expo_wgs.geometry.centroid
gdf_expo_pts = gdf_expo_wgs.set_geometry('centroid')

try:
    from shapely.ops import nearest_points
    # Jointure par proximité (nécessite geopandas >= 0.10)
    gdf_joined = gpd.sjoin_nearest(
        gdf_expo_pts[['jrc_category','floor_area_m2','max_damage_total_eur','centroid']],
        gdf_dvf[['prix_m2','geometry']],
        how='left', max_distance=0.01  # ~1km
    ).drop_duplicates(subset=gdf_expo_pts.index.name or gdf_expo_pts.index.tolist())

    gdf_expo_wgs['prix_m2_dvf'] = gdf_joined['prix_m2'].values[:len(gdf_expo_wgs)]
    gdf_expo_wgs['valeur_marchande_eur'] = (
        gdf_expo_wgs['floor_area_m2'] * gdf_expo_wgs['prix_m2_dvf']
    ).round(0)
    print("✅ Jointure BD TOPO × DVF réussie")
    print(f"Valeur marchande totale exposée (zone test) : {gdf_expo_wgs['valeur_marchande_eur'].sum()/1e6:.1f} M€")
    print(f"Valeur JRC totale exposée (zone test)       : {gdf_expo_wgs['max_damage_total_eur'].sum()/1e6:.1f} M€")
except Exception as e:
    print(f"⚠️  Jointure non disponible : {e}")
    print("→ Utilisation des valeurs JRC comme proxy de valeur")

In [ ]:
# ── 6.2 Tableau récapitulatif final ──────────────────────────────────────────
summary = pd.DataFrame([
    {"Couche"         : "BD TOPO Bâtiments",
     "Source"         : "IGN",
     "Nb entités"     : len(gdf_bdtopo),
     "Valeur exposée" : f"{gdf_exposure['max_damage_total_eur'].sum()/1e6:.1f} M€",
     "Statut"         : "✅ Chargé"},
    {"Couche"         : "FILOSOFI Population",
     "Source"         : "INSEE",
     "Nb entités"     : len(gdf_filo),
     "Valeur exposée" : f"{gdf_filo['Ind'].sum():,.0f} hab.",
     "Statut"         : "✅ Chargé"},
    {"Couche"         : "DVF Transactions",
     "Source"         : "DGFiP",
     "Nb entités"     : len(gdf_dvf),
     "Valeur exposée" : f"{gdf_dvf['valeur_fonciere'].sum()/1e6:.0f} M€ (transact.)",
     "Statut"         : "✅ Chargé"},
])
print("=== Couche d'exposition — Résumé ===")
display(summary)

# Sauvegarde
gdf_exposure.to_file(f"{DATA_DIR}/exposure_layer.gpkg", driver='GPKG')
print(f"\n✅ Couche exposition sauvegardée : {DATA_DIR}/exposure_layer.gpkg")
print("\n→ Prochaine étape : 05a_jrc_damage_curves.ipynb (courbes de vulnérabilité)")
print("→ Puis : 06a_risk_mapping.ipynb (H × V × E = risque)")